# NaiveBayes: Melhor modelo encontrado

## Melhor classificador encontrado

**COPIA E COLA OS PARAMETROS DO MELHOR MODELO NO EP2 AQUI, IGUAL O EXEMPLO AI EMBAIXO, E APAGA O EXEMPLO**

Exemplo:

O melhor classificador encontrado pelas pipelines foi -->    feature=WORD-NGram + solver=lbfgs

Melhor acucácia encontrada:  0.8565775957013552

Melhores parametros encontrados:  {'classifier__C': 1, 'classifier__max_iter': 1000, 'classifier__penalty': 'l2', 
'classifier__solver': 'lbfgs', 'kbest__k': 10000, 'vect__analyzer': 'word', 'vect__ngram_range': (1, 2)}

# Classificador: Word-NGram + solver-lbfgs

## Imports

In [ ]:
import pandas as pd
import numpy as np

import os
import sys
from pathlib import Path

filedir = Path(os.getcwd())
base_path = filedir.resolve().parents[3]
sys.path.append(str(base_path))

from Lib.utils import printhello
printhello()

from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectKBest, chi2
from imblearn.under_sampling import RandomUnderSampler

import warnings
from sklearn.exceptions import ConvergenceWarning

# Ignora warnings de ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Ignora UserWarnings específicos de l1_ratio etc
warnings.filterwarnings("ignore", category=UserWarning)

### Import dataframe

In [ ]:
sep = ";"
dec = ","
quotech = "\""
encoding = "latin-1"


EP_dir = "EP2"
CSV_input_name = "ep2-train.csv"
path_to_archive = f"../../../../Traindata/{EP_dir}/{CSV_input_name}"


do_print = True
if do_print:
    print(f"Path to csv input is:  {path_to_archive}")

In [ ]:
random_state = 12345
best_models_list = []

In [ ]:
df = pd.read_csv(path_to_archive, na_values=['na'],
sep=sep,
decimal=dec,
quotechar=quotech,
encoding=encoding,
encoding_errors='strict')
print(df.shape)
print(df.columns)

## Embaralhamento dos dados

In [ ]:
print("Shape antes do shuffle:", df.shape)

df = df.sample(frac=1, random_state=random_state).reset_index(drop=True) #NAO MUDE random_state, essa variavel DEVE valer 12345, ou QUEBRARÁ REPRODUTIBILIDADE dos experimentos

print("Shape depois do shuffle:", df.shape)

## Limpeza dos dados

In [ ]:
from Lib.utils import clean_text
#def clean_text(text, do_lowercase: bool, rem_emails: bool, rem_urls: bool, normalize_whitespaces: bool):

df['req_text_cleaned'] = df['req_text'].apply(lambda row_text: clean_text(
        row_text, 
        do_lowercase=True, 
        rem_emails=True, 
        rem_urls=True, 
        normalize_whitespaces=True
    ))

df['req_text'] = df['req_text_cleaned']
df = df.drop(columns=['req_text_cleaned']) # Remove a coluna temporária

## Modelo

In [ ]:

pipeline_BEST = Pipeline([
    ('vect', CountVectorizer()),
    ('kbest', SelectKBest(score_func=chi2)),
    ('classifier', MultinomialNB()),
]) 

parameters_BEST = { 
    
}


classifier_BEST = GridSearchCV(pipeline_BEST, parameters_BEST, 
                                       cv=10, n_jobs=2, scoring="accuracy", verbose=1, error_score = np.nan)
classifier_BEST.fit(df["req_text"].fillna(""), df["profession"].values)

print("Acurácia média:", classifier_BEST.best_score_)

## Controle de Reprodutibilidade

A sessão de código abaixo deve ser configurada com os parâmetros da solução ótima obtida através de experimentação, e executada somente para validar a reprodutibilidade da entrega.

Saída limpa significa que não houve quebra de reprodutibilidade com os resultados desenvolvidos ao longo do trabalho.

In [ ]:
import math

BEST_accuracy = classifier_BEST.best_score_
control_accuracy = 0.8565775957013552


if not math.isclose(BEST_accuracy, control_accuracy, rel_tol=1e-11, abs_tol=1e-11):
    print(f"Quebra de controle da Acurácia: BEST Acc. é {BEST_accuracy} e Control Acc é {control_accuracy}")